In [ ]:
import os

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
# 1. Initialize the client (Persistent saves to disk)
client = chromadb.PersistentClient(path="./RAG_chroma_db")

# 2. Create or get a collection (like a "table" in SQL)
# By default, it uses 'all-MiniLM-L6-v2' to create embeddings locally
collection = client.get_or_create_collection(name="youtube_transcripts", metadata={"hnsw:batch_size": 10000})

In [ ]:
# import chromadb
# # This connects to the Docker container, bypassing your local binary issues
# client = chromadb.HttpClient(host='localhost', port=8000)
# test_collection = client.get_or_create_collection("test")



In [ ]:
# test_collection

# Step 1 - Indexing

## Step 1a - Document Ingestion

In [ ]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
api = YouTubeTranscriptApi()
try:
    # If you don’t care which language, this returns the “best” one

    transcript_list = api.fetch(video_id, languages=["en"])
    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [ ]:
transcript_list

## Step 1b - Text Splitting

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript], metadatas= [{ 'video_id': video_id}])

In [ ]:
# len(chunks)

In [ ]:
# chunks[100]

## Step 1c - Embedding Generation

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# chunks[100].metadata, chunks[100].page_content

In [ ]:
response = embeddings.embed_documents([chunk.page_content for chunk in chunks])
# 2. Create unique IDs and metadata for each chunk
ids = [str(i) for i in range(len(chunks))]
metadatas = [chunk.metadata | {"chunk_index": i} for i, chunk in enumerate(chunks)]

In [ ]:
# final_embeddings = [list(e) for e in response] if hasattr(response[0], '__iter__') else [list(response)]

In [ ]:
docs = [chunk.page_content for chunk in chunks]

In [ ]:
collection.add(
embeddings=response[0],
documents=docs[0],
# metadatas=metadatas[0],
ids=ids[0])

In [ ]:
# This WILL NOT crash your kernel because the work happens in Docker
test_collection.add(ids=["1"], documents=["test"], embeddings=[[0.1]*1536])

In [ ]:
import chromadb

# Connect to the server you just started
client = chromadb.HttpClient(host='127.0.0.1', port=8000)

# Verify the connection
try:
    print(f"Chroma Heartbeat: {client.heartbeat()}")
    
    # Get your collection
    collection = client.get_or_create_collection(name="test")
    
    # Add a test document (using your 1536 dimension size)
    collection.add(
        ids=["doc_1"],
        documents=["This is a test document"],
        embeddings=[[0.1] * 1536]
    )
    print("✅ Successfully added data to Docker!")
    
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
batch_size = 5  # Start very low to be safe
for i in range(0, len(docs), batch_size):
    collection.add(
        embeddings=response[i : i + batch_size],
        documents=docs[i : i + batch_size],
        metadatas=metadatas[i : i + batch_size],
        ids=ids[i : i + batch_size]
    )

In [ ]:
embeddings.

Step 1d - Storing in Vector Store

In [ ]:
def process_transcript(video_id, chunks):
    """
    chunks: List of strings (e.g., from your transcript splitter)
    """
    # 1. Generate embeddings in a single batch call
    response = client.embeddings.create(
        input=chunks,
        model="text-embedding-3-small"
    )
    
    # Extract the vectors
    embeddings = [item.embedding for item in response.data]
    
    # 2. Create unique IDs and metadata for each chunk
    ids = [f"{video_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"video_id": video_id, "chunk_index": i} for i in range(len(chunks))]
    
    # 3. Add to Chroma store
    collection.add(
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Successfully stored {len(chunks)} chunks for video: {video_id}")

# Example Usage:
# my_chunks = ["snippet 1...", "snippet 2...", "snippet 3..."]
# process_transcript("dQw4w9WgXcQ", my_chunks)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['e99d2de5-59a1-4b65-8c1b-4b81005f1166'])

# Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is deepmind')

# Step 3 - Augmentation

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

# Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

In [ ]:
import os
import chromadb
from langchain_openai import OpenAIEmbeddings

# 1. Initialize same embeddings model used for storage
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Connect to your existing Docker container
client = chromadb.HttpClient(host='127.0.0.1', port=8000)

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
api = YouTubeTranscriptApi()
try:
    # If you don’t care which language, this returns the “best” one

    transcript_list = api.fetch(video_id, languages=["en"])
    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript], metadatas= [{ 'video_id': video_id}])

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
response = embeddings.embed_documents([chunk.page_content for chunk in chunks])
# 2. Create unique IDs and metadata for each chunk
ids = [str(i) for i in range(len(chunks))]
metadatas = [chunk.metadata | {"chunk_index": i} for i, chunk in enumerate(chunks)]

In [ ]:
docs = [chunk.page_content for chunk in chunks]

In [ ]:
collection = client.get_or_create_collection("youtube_transcripts")

collection.add(
embeddings=response,
documents=docs,
metadatas=metadatas,
ids=ids)

In [ ]:
# 3. Create a query vector from your question
user_question = "What is deepmind"
query_vector = embeddings_model.embed_query(user_question)

# 4. Search Chroma
results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,  # Number of matching chunks to return
    include=["documents", "metadatas", "distances"]
)

In [ ]:
results

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
user_question = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"

query_vector = embeddings_model.embed_query(user_question)

# 4. Search Chroma
results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,  # Number of matching chunks to return
    include=["documents", "metadatas", "distances"]
)

In [ ]:
results

In [ ]:
results['documents'][0]


In [ ]:
context_text = "\n\n".join(doc for doc in results['documents'][0])

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
print(final_prompt)

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

In [ ]:
import sqlite3

# Path to the file in your project folder
db_path = "./vector_store/chroma.sqlite3"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Example: Get all your collection names
cursor.execute("SELECT name FROM collections")
print(cursor.fetchall())

conn.close()

In [ ]:
import sqlite3
import pandas as pd

# Path to the file in your project folder
db_path = "./vector_store/chroma.sqlite3"
# 1. Connect to the copied database file
conn = sqlite3.connect(db_path)

# 2. Get a list of all table names to verify the structure
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in Database:\n", tables)

# 3. View your collections
collections = pd.read_sql_query("SELECT * FROM collections;", conn)
print("\nYour Collections:\n", collections)

# 4. View specific document text and metadata
# Joining metadata and text tables (structure may vary slightly by Chroma version)
query = """
SELECT 
    m.id, 
    m.key, 
    m.string_value,
    e.embedding_id
FROM embedding_metadata m
JOIN embeddings e ON m.id = e.id
LIMIT 10;
"""
data_sample = pd.read_sql_query(query, conn)
print("\nSample Data:\n", data_sample)

conn.close()
